# Markov-Switching GARCH для моделирования волатильности в кризисные периоды

Активы: S&P 500 (COVID-19) и Bitcoin (крипто-крахи: 2018, COVID-март-2020, Terra/Luna май-2022, FTX ноябрь-2022).

По каждому активу берём максимум доступной дневной истории отдельно

## Структура ноутбука (будет заполняться)

0. Setup
1. Загрузка данных
2. EDA: графики цен и доходностей, описательная статистика, тест Харке-Бера
3. Стационарность: ADF, KPSS
4. Кластеризация волатильности: ACF/PACF квадратов доходностей
5. Тест на ARCH-эффекты (LM/Engle test)
6. Базовые модели: ARCH, GARCH, подбор порядка по AIC/BIC
7. GJR-GARCH с t-распределением Стьюдента
8. Марковская цепь: игрушечный пример
9. Markov-Switching GARCH: построение модели
10. Вероятности режимов (фильтрованные / сглаженные)
11. Сравнение моделей (AIC/BIC/log-likelihood)
12. VaR и тест Купика
13. Повтор шагов 2-12 для Bitcoin
14. Итоговое сравнение двух кейсов
15. Экспорт финальных графиков и таблиц


## 0. Setup

In [10]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf

from scipy import stats

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_lm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from arch import arch_model

%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 4)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)


## 1. Загрузка данных: S&P 500 и Bitcoin

Качаем каждый актив отдельно, чтобы не обрезать более длинный ряд (S&P 500) по дате начала более короткого (Bitcoin). При повторном запуске данные читаются из локального CSV в папке `data/`

In [11]:
def load_or_download(ticker, start, path):
    if os.path.exists(path):
        s = pd.read_csv(path, index_col=0, parse_dates=True).iloc[:, 0]
        print(f'{ticker}: загружено из кэша {path}')
    else:
        s = yf.download(ticker, start=start, auto_adjust=False)['Close'].dropna()
        s.to_csv(path)
        print(f'{ticker}: скачано с Yahoo Finance и сохранено в {path}')
    return s


sp500_prices = load_or_download('^GSPC', '1990-01-01', os.path.join(DATA_DIR, 'sp500_prices.csv'))
btc_prices   = load_or_download('BTC-USD', '2010-01-01', os.path.join(DATA_DIR, 'btc_prices.csv'))

print()
print(f'S&P 500: {sp500_prices.index[0].date()} - {sp500_prices.index[-1].date()}, {len(sp500_prices)} наблюдений')
print(f'Bitcoin: {btc_prices.index[0].date()} - {btc_prices.index[-1].date()}, {len(btc_prices)} наблюдений')


^GSPC: загружено из кэша data/sp500_prices.csv
BTC-USD: загружено из кэша data/btc_prices.csv

S&P 500: 1990-01-02 - 2026-09-09, 9239 наблюдений
Bitcoin: 2014-09-17 - 2026-09-10, 4377 наблюдений
